## Import library

In [1]:
import yaml
import os
import boto3
import pandas as pd
from pprint import pprint
import shutil
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelBinarizer

import warnings
warnings.filterwarnings("ignore")

In [2]:
def read_yaml_file(path, file):
    with open(os.path.join(path, file)) as f:
        try:
            content = yaml.safe_load(f)
        except yaml.YAMLError as e:
            raise e
    
    return content


CONFIG_PATH = os.path.join("..", "src", "config")

In [3]:
credentials = read_yaml_file(path=CONFIG_PATH, file="credentials.yaml")
settings = read_yaml_file(path=CONFIG_PATH, file="settings.yaml")

AWS_ACCESS_KEY = credentials['AWS_ACCESS_KEY']
AWS_SECRET_KEY = credentials['AWS_SECRET_KEY']
S3_NAME = credentials['S3']

ARTIFACTS_OUTPUT_PATH = settings['ARTIFACTS_PATH']
FEATURES_OUTPUT_PATH = settings['FEATURES_PATH']
RAW_FILE_PATH = os.path.join(settings["DATA_PATH"], settings["RAW_FILE_NAME"])
PROCESSED_RAW_FILE = "Preprocessed_" + settings["RAW_FILE_NAME"]
PROCESSED_RAW_FILE_PATH = os.path.join(settings["DATA_PATH"], PROCESSED_RAW_FILE)

In [4]:
settings["RAW_FILE_NAME"]

'Original_ObesityDataSet.csv'

In [5]:
RAW_FILE_PATH = f"../{RAW_FILE_PATH}"
PROCESSED_RAW_FILE_PATH = f"../{PROCESSED_RAW_FILE_PATH}"
ARTIFACTS_OUTPUT_PATH = f"../{ARTIFACTS_OUTPUT_PATH}"
FEATURES_OUTPUT_PATH = f"../{FEATURES_OUTPUT_PATH}"

In [6]:
# Inittials S3 client (for low-level operations)
s3_client = boto3.client(
    service_name = 's3',
    aws_access_key_id = AWS_ACCESS_KEY,
    aws_secret_access_key = AWS_SECRET_KEY
)
if not os.path.exists(RAW_FILE_PATH):
    s3_client.download_file(S3_NAME, settings["RAW_FILE_NAME"], RAW_FILE_PATH)

## Data cleaning

In [8]:
df = pd.read_csv(RAW_FILE_PATH)
df.drop('id', axis=1, inplace=True)
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


### Removing Duplicates

In [32]:
df = df.drop_duplicates(keep='first')
pprint(f"Data Shape: {df.shape}")

'Data Shape: (2087, 17)'


### Transform Height units to Cetimeters

In [33]:
df['Height'] *= 100

### Removing Outliers

In [34]:
df.describe()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000
mean,24.353090,170.267412,86.858730,2.421466,2.701179,2.004749,1.012812,0.663035
std,6.368801,9.318594,26.190847,0.534737,0.764614,0.608284,0.853475,0.608153
min,14.000000,145.000000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,19.915937,163.017850,66.000000,2.000000,2.697467,1.590922,0.124505,0.000000
50%,22.847618,170.158400,83.101100,2.396265,3.000000,2.000000,1.000000,0.630866
75%,26.000000,176.949150,108.015907,3.000000,3.000000,2.466193,1.678102,1.000000
max,61.000000,198.000000,173.000000,3.000000,4.000000,3.000000,3.000000,2.000000


In [35]:
# calculating the upper and lower limits
Q1 = df["Age"].quantile(0.25)
Q3 = df["Age"].quantile(0.75)
# threshold = 1.5
threshold = 3.0
IQR = Q3 - Q1

pprint(f"Dataset shape before removing the outliers: {df.shape}")

# removing the data samples that exceeds the upper or lower limits
df = df[~((df["Age"] >= (Q3 + threshold * IQR)) | (df["Age"] <= (Q1 - threshold * IQR)))]
pprint(f"Dataset shape after removing the outliers: {df.shape}")

'Dataset shape before removing the outliers: (2087, 17)'
'Dataset shape after removing the outliers: (2070, 17)'


## Creating New Features

### Body Mass Index (BMI)

In [36]:
df["BMI"] = df["Weight"] / (df["Height"] ** 2)

### Ideal Number of Main Meals? (INMM)

In [37]:
df["INMM"] = df["NCP"] == 3
df["INMM"] = df["INMM"].astype(int)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2070 entries, 0 to 2110
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          2070 non-null   object 
 1   Age                             2070 non-null   float64
 2   Height                          2070 non-null   float64
 3   Weight                          2070 non-null   float64
 4   family_history_with_overweight  2070 non-null   object 
 5   FAVC                            2070 non-null   object 
 6   FCVC                            2070 non-null   float64
 7   NCP                             2070 non-null   float64
 8   CAEC                            2070 non-null   object 
 9   SMOKE                           2070 non-null   object 
 10  CH2O                            2070 non-null   float64
 11  SCC                             2070 non-null   object 
 12  FAF                             2070 no

### Transforming `Age` Column Into a Categorical Column

Reducing the impact of outliers in the `Age` column using *Quantile Bucketing*

In [39]:
values, bins = pd.qcut(x=df["Age"], q=4, retbins=True, labels=["q1", "q2", "q3", "q4"])

In [40]:
print(type(bins))
print(bins)

<class 'numpy.ndarray'>
[14.         19.87459025 22.8097375  26.         44.        ]


In [41]:

bins = np.concatenate(([-np.inf], bins[1:-1], [np.inf]))

df["Age"] = values
df["Age"] = df["Age"].astype("object")
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,BMI,INMM
0,Female,q2,162.0,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight,0.002439,1
1,Female,q2,152.0,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight,0.002424,1
2,Male,q3,180.0,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight,0.002377,1
3,Male,q4,180.0,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I,0.002685,1
4,Male,q2,178.0,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II,0.002834,0


In [42]:
print(type(bins))
print(bins)

<class 'numpy.ndarray'>
[       -inf 19.87459025 22.8097375  26.                 inf]


### Transforming `INMM` into Categorical Columns


In [43]:
df["INMM"] = df["INMM"].astype("object")
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,BMI,INMM
0,Female,q2,162.0,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight,0.002439,1
1,Female,q2,152.0,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight,0.002424,1
2,Male,q3,180.0,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight,0.002377,1
3,Male,q4,180.0,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I,0.002685,1
4,Male,q2,178.0,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II,0.002834,0


### Spliting data into training and validation sets

In [44]:
# df.drop(["NObeyesdad"], axis=1, inplace=True)

In [45]:
X = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"].values

In [46]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, stratify=y ,random_state=42)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)

pprint(f"Train set shape: {X_train.shape} and {y_train.shape}")
pprint(f"Validation set shape: {X_val.shape} and {y_val.shape}")

'Train set shape: (1656, 18) and (1656,)'
'Validation set shape: (414, 18) and (414,)'


### Transform numerical columns (Log + 1 tranformation)

In [47]:
numerical_columns = df.select_dtypes(include=["number"]).columns.to_list()

for col in numerical_columns:
    X_train[col] = np.log1p(X_train[col])
    X_val[col] = np.log1p(X_val[col])

In [48]:
numerical_columns

['Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'BMI']

### Scaling the numerical columns

In [49]:
pprint("Training set skewness before scaling:")
pprint(X_train[numerical_columns].skew())
pprint("Validation set skewness before scaling:")
pprint(X_val[numerical_columns].skew())

'Training set skewness before scaling:'
Height   -0.166453
Weight   -0.299523
FCVC     -0.817904
NCP      -1.479285
CH2O     -0.451897
FAF      -0.089907
TUE       0.128768
BMI       0.140661
dtype: float64
'Validation set skewness before scaling:'
Height   -0.053020
Weight   -0.306561
FCVC     -0.865311
NCP      -1.489237
CH2O     -0.464923
FAF      -0.056350
TUE       0.102569
BMI       0.107478
dtype: float64


**Fit:** Use `fit()` on your training data to calculate the mean and standard deviation for each feature. <br>
**Transform:** Use `transform()` to apply the scaling to your training and test data. It's crucial to use the same fitted scaler for both to ensure consistency.


- *Outliers:* `StandardScaler` can be sensitive to outliers
- *Data leakage:* Always fit the `StandardScaler` only on the training data and then apply the learned transformation to both training and test sets.

In [50]:
scalers = {}

for col in numerical_columns:
    sc = StandardScaler()

    sc.fit(X_train[col].to_numpy().reshape(-1,1))

    X_train[col] = sc.transform(X_train[col].to_numpy().reshape(-1,1))
    X_val[col] = sc.transform(X_val[col].to_numpy().reshape(-1,1))
    scalers[col] = sc

In [65]:
type(scalers)

dict

In [51]:
# scalers = {}

# for col in numerical_columns:
#     sc = StandardScaler()

#     sc.fit(df[col].to_numpy().reshape(-1,1))

#     df[col] = sc.transform(df[col].to_numpy().reshape(-1,1))

#     scalers[col] = sc

In [52]:
# df

In [69]:
scalers

{'Height': StandardScaler(),
 'Weight': StandardScaler(),
 'FCVC': StandardScaler(),
 'NCP': StandardScaler(),
 'CH2O': StandardScaler(),
 'FAF': StandardScaler(),
 'TUE': StandardScaler(),
 'BMI': StandardScaler()}

In [75]:
key = list(scalers.keys())[0]
key

isinstance(scalers[key], StandardScaler)

True

In [54]:
pprint("Training set skewness after scaling:")
pprint(X_train[numerical_columns].skew())
print()
pprint("Validation set skewness after scaling:")
pprint(X_val[numerical_columns].skew())

'Training set skewness after scaling:'
Height   -0.166453
Weight   -0.299523
FCVC     -0.817904
NCP      -1.479285
CH2O     -0.451897
FAF      -0.089907
TUE       0.128768
BMI       0.140661
dtype: float64

'Validation set skewness after scaling:'
Height   -0.053020
Weight   -0.306561
FCVC     -0.865311
NCP      -1.489237
CH2O     -0.464923
FAF      -0.056350
TUE       0.102569
BMI       0.107478
dtype: float64


### Encoding categorical columns

In [55]:
categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.to_list()

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist', min_frequency=20)
encoder.fit(X_train[categorical_columns])

train_encoder_df  = pd.DataFrame(
    data=encoder.transform(X_train[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

val_encoder_df  = pd.DataFrame(
    data=encoder.transform(X_val[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

new_train_df = pd.concat([train_encoder_df, X_train.drop(categorical_columns, axis=1)], axis=1)
new_val_df = pd.concat([val_encoder_df, X_val.drop(categorical_columns, axis=1)], axis=1)

X_train = new_train_df.values.copy()
X_val = new_val_df.values.copy()

In [76]:
type(encoder)
isinstance(encoder, OneHotEncoder)

True

In [58]:
encoder.get_feature_names_out()

array(['Gender_Male', 'Age_q2', 'Age_q3', 'Age_q4',
       'family_history_with_overweight_yes', 'FAVC_yes',
       'CAEC_Frequently', 'CAEC_Sometimes', 'CAEC_no', 'SMOKE_yes',
       'SCC_yes', 'CALC_Sometimes', 'CALC_no', 'CALC_infrequent_sklearn',
       'MTRANS_Public_Transportation', 'MTRANS_Walking',
       'MTRANS_infrequent_sklearn', 'INMM_1'], dtype=object)

### Encoding the labels

In [59]:
label_encoder = LabelBinarizer(sparse_output=False)
label_encoder.fit(y_train)

original_y_train = y_train.copy()
original_y_valid = y_val.copy()

y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)

In [60]:
type(label_encoder)

sklearn.preprocessing._label.LabelBinarizer

In [61]:
pprint(f"Train set shape: {X_train.shape} and {y_train.shape}")
pprint(f"Validation set shape: {X_val.shape} and {y_val.shape}")

'Train set shape: (1656, 26) and (1656, 7)'
'Validation set shape: (414, 26) and (414, 7)'


In [62]:
label_encoder.classes_

array(['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I',
       'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I',
       'Overweight_Level_II'], dtype='<U19')

### Saving the Artifacts

In [ ]:
# saving the artifacts locally
os.makedirs(ARTIFACTS_OUTPUT_PATH, exist_ok=True)
os.makedirs(FEATURES_OUTPUT_PATH, exist_ok=True)

with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'features_encoder.pkl'), 'wb') as f:
    pickle.dump(encoder, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'qcut_bins.pkl'), 'wb') as f:
    pickle.dump(bins, f)

with open(os.path.join(FEATURES_OUTPUT_PATH, 'X_train.pkl'), 'wb') as f:
    pickle.dump(X_train, f)
with open(os.path.join(FEATURES_OUTPUT_PATH, 'X_val.pkl'), 'wb') as f:
    pickle.dump(X_val, f)
with open(os.path.join(FEATURES_OUTPUT_PATH, 'y_train.pkl'), 'wb') as f:
    pickle.dump(y_train, f)
with open(os.path.join(FEATURES_OUTPUT_PATH, 'y_val.pkl'), 'wb') as f:
    pickle.dump(y_val, f)

In [ ]:
# saving the preprocessed dataset locally
new_train_df['NObeyesdad'] = original_y_train
new_val_df['NObeyesdad'] = original_y_valid

preprocessed_data = pd.concat([new_train_df, new_val_df])
preprocessed_data.to_csv(PROCESSED_RAW_FILE_PATH, index=False, sep=",")

In [ ]:
def upload_folder_s3(root_path: str, s3_folder_prefix=""):
    try:
        for root, dirs, files in os.walk(root_path):
            for file_name in files:
                local_path = os.path.join(root, file_name)
                relative_path = os.path.relpath(local_path, root_path)

                if s3_folder_prefix:
                    s3_key = f"{s3_folder_prefix}/{relative_path}".replace("\\", "/")
                else:
                    s3_key = relative_path.replace("\\", "/")

                s3_client.upload_file(local_path, S3_NAME, s3_key)
                print(f"✅ Uploaded: {local_path} → s3://{S3_NAME}/{s3_key}")
    except Exception as err:
        print(f"❌ Upload failed: {err}")

if os.path.exists(ARTIFACTS_OUTPUT_PATH):
    upload_folder_s3(ARTIFACTS_OUTPUT_PATH, s3_folder_prefix="artifacts")

if os.path.exists(FEATURES_OUTPUT_PATH):
    upload_folder_s3(FEATURES_OUTPUT_PATH, s3_folder_prefix="features")

# sending preprocessed dataset saved locally to the aws s3 bucket
s3_client.upload_file(
    PROCESSED_RAW_FILE_PATH,
    credentials["S3"],
    PROCESSED_RAW_FILE
)

✅ Uploaded: ..//models/artifacts\features_encoder.pkl → s3://bucket6502-aws/artifacts/features_encoder.pkl
✅ Uploaded: ..//models/artifacts\label_encoder.pkl → s3://bucket6502-aws/artifacts/label_encoder.pkl
✅ Uploaded: ..//models/artifacts\qcut_bins.pkl → s3://bucket6502-aws/artifacts/qcut_bins.pkl
✅ Uploaded: ..//models/artifacts\scalers.pkl → s3://bucket6502-aws/artifacts/scalers.pkl
✅ Uploaded: ..//models/features\X_train.pkl → s3://bucket6502-aws/features/X_train.pkl
✅ Uploaded: ..//models/features\X_val.pkl → s3://bucket6502-aws/features/X_val.pkl
✅ Uploaded: ..//models/features\y_train.pkl → s3://bucket6502-aws/features/y_train.pkl
✅ Uploaded: ..//models/features\y_val.pkl → s3://bucket6502-aws/features/y_val.pkl


In [ ]:
# if os.path.exists(ARTIFACTS_OUTPUT_PATH):
#     shutil.rmtree(ARTIFACTS_OUTPUT_PATH)

# if os.path.exists(FEATURES_OUTPUT_PATH):
#     shutil.rmtree(FEATURES_OUTPUT_PATH)

# if os.path.exists(RAW_FILE_PATH):
#     os.remove(RAW_FILE_PATH)

# if os.path.exists(PROCESSED_RAW_FILE_PATH):
#     os.remove(PROCESSED_RAW_FILE_PATH)

In [ ]:
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'features_encoder.pkl'), 'wb') as f:
    pickle.dump(encoder, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)
with open(os.path.join(ARTIFACTS_OUTPUT_PATH, 'qcut_bins.pkl'), 'wb') as f:
    pickle.dump(bins, f)

In [ ]:
df = pd.read_csv("D:\My_project\e2e_ml\data\Original_ObesityDataSet.csv")
df.drop(["id", "NObeyesdad"], inplace=True, axis=1)

In [ ]:
df = df.head(10)

In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation
1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile
2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation
3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation
4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation
5,Male,18.128249,1.748524,51.552595,yes,yes,2.919751,3.000000,Sometimes,no,2.137550,no,1.930033,1.000000,Sometimes,Public_Transportation
6,Male,29.883021,1.754711,112.725005,yes,yes,1.991240,3.000000,Sometimes,no,2.000000,no,0.000000,0.696948,Sometimes,Automobile
7,Male,29.891473,1.750150,118.206565,yes,yes,1.397468,3.000000,Sometimes,no,2.000000,no,0.598655,0.000000,Sometimes,Automobile
8,Male,17.000000,1.700000,70.000000,no,yes,2.000000,3.000000,Sometimes,no,3.000000,yes,1.000000,1.000000,no,Public_Transportation
9,Female,26.000000,1.638836,111.275646,yes,yes,3.000000,3.000000,Sometimes,no,2.632253,no,0.000000,0.218645,Sometimes,Public_Transportation


In [ ]:
df['Height'] *= 100

In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Male,24.443011,169.9998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation
1,Female,18.000000,156.0000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile
2,Female,18.000000,171.1460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation
3,Female,20.952737,171.0730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation
4,Male,31.641081,191.4186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation
5,Male,18.128249,174.8524,51.552595,yes,yes,2.919751,3.000000,Sometimes,no,2.137550,no,1.930033,1.000000,Sometimes,Public_Transportation
6,Male,29.883021,175.4711,112.725005,yes,yes,1.991240,3.000000,Sometimes,no,2.000000,no,0.000000,0.696948,Sometimes,Automobile
7,Male,29.891473,175.0150,118.206565,yes,yes,1.397468,3.000000,Sometimes,no,2.000000,no,0.598655,0.000000,Sometimes,Automobile
8,Male,17.000000,170.0000,70.000000,no,yes,2.000000,3.000000,Sometimes,no,3.000000,yes,1.000000,1.000000,no,Public_Transportation
9,Female,26.000000,163.8836,111.275646,yes,yes,3.000000,3.000000,Sometimes,no,2.632253,no,0.000000,0.218645,Sometimes,Public_Transportation


In [ ]:
df['BMI'] = df['Weight'] / (df['Height'] ** 2) 

In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,BMI
0,Male,24.443011,169.9998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,0.002826
1,Female,18.000000,156.0000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,0.002342
2,Female,18.000000,171.1460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,0.001713
3,Female,20.952737,171.0730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,0.004486
4,Male,31.641081,191.4186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,0.002560
5,Male,18.128249,174.8524,51.552595,yes,yes,2.919751,3.000000,Sometimes,no,2.137550,no,1.930033,1.000000,Sometimes,Public_Transportation,0.001686
6,Male,29.883021,175.4711,112.725005,yes,yes,1.991240,3.000000,Sometimes,no,2.000000,no,0.000000,0.696948,Sometimes,Automobile,0.003661
7,Male,29.891473,175.0150,118.206565,yes,yes,1.397468,3.000000,Sometimes,no,2.000000,no,0.598655,0.000000,Sometimes,Automobile,0.003859
8,Male,17.000000,170.0000,70.000000,no,yes,2.000000,3.000000,Sometimes,no,3.000000,yes,1.000000,1.000000,no,Public_Transportation,0.002422
9,Female,26.000000,163.8836,111.275646,yes,yes,3.000000,3.000000,Sometimes,no,2.632253,no,0.000000,0.218645,Sometimes,Public_Transportation,0.004143


In [ ]:
df['INMM'] = (df['NCP'] == 3).astype(int).astype("object")

In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,BMI,INMM
0,Male,24.443011,169.9998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,0.002826,0
1,Female,18.000000,156.0000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,0.002342,1
2,Female,18.000000,171.1460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,0.001713,0
3,Female,20.952737,171.0730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,0.004486,1
4,Male,31.641081,191.4186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,0.002560,0
5,Male,18.128249,174.8524,51.552595,yes,yes,2.919751,3.000000,Sometimes,no,2.137550,no,1.930033,1.000000,Sometimes,Public_Transportation,0.001686,1
6,Male,29.883021,175.4711,112.725005,yes,yes,1.991240,3.000000,Sometimes,no,2.000000,no,0.000000,0.696948,Sometimes,Automobile,0.003661,1
7,Male,29.891473,175.0150,118.206565,yes,yes,1.397468,3.000000,Sometimes,no,2.000000,no,0.598655,0.000000,Sometimes,Automobile,0.003859,1
8,Male,17.000000,170.0000,70.000000,no,yes,2.000000,3.000000,Sometimes,no,3.000000,yes,1.000000,1.000000,no,Public_Transportation,0.002422,1
9,Female,26.000000,163.8836,111.275646,yes,yes,3.000000,3.000000,Sometimes,no,2.632253,no,0.000000,0.218645,Sometimes,Public_Transportation,0.004143,1


In [ ]:
df['Age'] = pd.cut(df['Age'], bins=bins, labels=["q1", "q2", "q3", "q4"], include_lowest=True).astype("object")

In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,BMI,INMM
0,Male,q3,169.9998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,0.002826,0
1,Female,q1,156.0000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,0.002342,1
2,Female,q1,171.1460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,0.001713,0
3,Female,q2,171.0730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,0.004486,1
4,Male,q4,191.4186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,0.002560,0
5,Male,q1,174.8524,51.552595,yes,yes,2.919751,3.000000,Sometimes,no,2.137550,no,1.930033,1.000000,Sometimes,Public_Transportation,0.001686,1
6,Male,q4,175.4711,112.725005,yes,yes,1.991240,3.000000,Sometimes,no,2.000000,no,0.000000,0.696948,Sometimes,Automobile,0.003661,1
7,Male,q4,175.0150,118.206565,yes,yes,1.397468,3.000000,Sometimes,no,2.000000,no,0.598655,0.000000,Sometimes,Automobile,0.003859,1
8,Male,q1,170.0000,70.000000,no,yes,2.000000,3.000000,Sometimes,no,3.000000,yes,1.000000,1.000000,no,Public_Transportation,0.002422,1
9,Female,q3,163.8836,111.275646,yes,yes,3.000000,3.000000,Sometimes,no,2.632253,no,0.000000,0.218645,Sometimes,Public_Transportation,0.004143,1


In [ ]:
for col in numerical_columns:
    df[col] = np.log1p(df[col])


In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,BMI,INMM
0,Male,q3,5.141662,4.414856,yes,yes,1.098612,1.382110,Sometimes,no,1.325369,no,0.000000,0.681314,Sometimes,Public_Transportation,0.002822,0
1,Female,q1,5.056246,4.060443,yes,yes,1.098612,1.386294,Frequently,no,1.098612,no,0.693147,0.693147,no,Automobile,0.002339,1
2,Female,q1,5.148343,3.935070,yes,yes,1.057976,0.880326,Sometimes,no,1.068283,no,0.623821,0.983420,no,Public_Transportation,0.001711,0
3,Female,q2,5.147919,4.884882,yes,yes,1.386294,1.386294,Sometimes,no,0.983598,no,0.903353,0.576725,Sometimes,Public_Transportation,0.004476,1
4,Male,q4,5.259673,4.551749,yes,yes,1.302821,1.089057,Sometimes,no,1.091872,no,1.087879,0.658411,Sometimes,Public_Transportation,0.002557,0
5,Male,q1,5.169645,3.961814,yes,yes,1.366028,1.386294,Sometimes,no,1.143442,no,1.075014,0.693147,Sometimes,Public_Transportation,0.001685,1
6,Male,q4,5.173157,4.733783,yes,yes,1.095688,1.386294,Sometimes,no,1.098612,no,0.000000,0.528831,Sometimes,Automobile,0.003654,1
7,Male,q4,5.170569,4.780858,yes,yes,0.874413,1.386294,Sometimes,no,1.098612,no,0.469163,0.000000,Sometimes,Automobile,0.003852,1
8,Male,q1,5.141664,4.262680,no,yes,1.098612,1.386294,Sometimes,no,1.386294,yes,0.693147,0.693147,no,Public_Transportation,0.002419,1
9,Female,q3,5.105240,4.720957,yes,yes,1.386294,1.386294,Sometimes,no,1.289853,no,0.000000,0.197740,Sometimes,Public_Transportation,0.004135,1


In [ ]:
for col in numerical_columns:
    df[col] = scalers[col].transform(df[col].to_numpy().reshape(-1, 1))

In [ ]:
df

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,BMI,INMM
0,Male,q3,0.021406,-0.080136,yes,yes,-0.770643,0.347358,Sometimes,no,1.117696,no,-1.344493,0.732265,Sometimes,Public_Transportation,-0.237933,0
1,Female,q1,-1.645782,-1.202305,yes,yes,-0.770643,0.365360,Frequently,no,0.055322,no,0.233001,0.764779,no,Automobile,-0.817362,1
2,Female,q1,0.151799,-1.599269,yes,yes,-1.019576,-1.811374,Sometimes,no,-0.086774,no,0.075226,1.562370,no,Public_Transportation,-1.571837,0
3,Female,q2,0.143520,1.408095,yes,yes,0.991650,0.365360,Sometimes,no,-0.483529,no,0.711396,0.444883,Sometimes,Public_Transportation,1.747828,1
4,Male,q4,2.324777,0.353304,yes,yes,0.480309,-0.913387,Sometimes,no,0.023744,no,1.131350,0.669335,Sometimes,Public_Transportation,-0.556561,0
5,Male,q1,0.567579,-1.514590,yes,yes,0.867503,0.365360,Sometimes,no,0.265354,no,1.102070,0.764779,Sometimes,Public_Transportation,-1.603579,1
6,Male,q4,0.636130,0.929675,yes,yes,-0.788557,0.365360,Sometimes,no,0.055322,no,-1.344493,0.313284,Sometimes,Automobile,0.761712,1
7,Male,q4,0.585618,1.078726,yes,yes,-2.144050,0.365360,Sometimes,no,0.055322,no,-0.276752,-1.139800,Sometimes,Automobile,0.998675,1
8,Male,q1,0.021428,-0.561967,no,yes,-0.770643,0.365360,Sometimes,no,1.403138,yes,0.233001,0.764779,no,Public_Transportation,-0.721596,1
9,Female,q3,-0.689502,0.889063,yes,yes,0.991650,0.365360,Sometimes,no,0.951302,no,-1.344493,-0.596466,Sometimes,Public_Transportation,1.338357,1


In [ ]:
encoded_df = pd.DataFrame(
    data=encoder.transform(df[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)
    
# Concatenate numerical and encoded categorical
df_final = pd.concat([encoded_df, df.drop(columns=categorical_columns,  axis=1)], axis=1)

In [ ]:
df_final

,Gender_Male,Age_q2,Age_q3,Age_q4,family_history_with_overweight_yes,FAVC_yes,CAEC_Frequently,CAEC_Sometimes,CAEC_no,SMOKE_yes,...,MTRANS_Walking,INMM_1,Height,Weight,FCVC,NCP,CH2O,FAF,TUE,BMI
0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.021406,-0.080136,-0.770643,0.347358,1.117696,-1.344493,0.732265,-0.237933
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,-1.645782,-1.202305,-0.770643,0.365360,0.055322,0.233001,0.764779,-0.817362
2,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.151799,-1.599269,-1.019576,-1.811374,-0.086774,0.075226,1.562370,-1.571837
3,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.143520,1.408095,0.991650,0.365360,-0.483529,0.711396,0.444883,1.747828
4,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,2.324777,0.353304,0.480309,-0.913387,0.023744,1.131350,0.669335,-0.556561
5,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.567579,-1.514590,0.867503,0.365360,0.265354,1.102070,0.764779,-1.603579
6,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.636130,0.929675,-0.788557,0.365360,0.055322,-1.344493,0.313284,0.761712
7,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.585618,1.078726,-2.144050,0.365360,0.055322,-0.276752,-1.139800,0.998675
8,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.021428,-0.561967,-0.770643,0.365360,1.403138,0.233001,0.764779,-0.721596
9,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,-0.689502,0.889063,0.991650,0.365360,0.951302,-1.344493,-0.596466,1.338357
